# Lung Disease V2.1: seed variance, EfficientNet, validation-fitted calibration, subgroups and Grad-CAM

Follows V2 (DenseNet121 0.801 and ResNet50 0.795 test mean AUROC, one seed). This run adds:
- **Two more DenseNet121 seeds** (1 and 2) to see how much the result moves between runs.
- **EfficientNet-B0** in the same pipeline.
- **Calibration fitted on the validation set** (per-class Platt scaling) and checked on the test set.
- **Patient-level bootstrap** intervals (resampling patients, not images).
- **Subgroup AUROC** by sex and age band.
- **Grad-CAM** heatmaps for a few test images.

Settings: Accelerator = GPU T4, Internet = On, input dataset `nih-chest-xrays/data`. Then "Save Version" with "Save & Run All".

Not a medical device. Research only, not for diagnosis.

In [1]:
import glob
import json
import os
import time
import traceback
from multiprocessing import Pool
from pathlib import Path

import numpy as np
import pandas as pd

SMOKE = False          # True = tiny run to check the pipeline
IMG = 224
BATCH = 128
MAX_EPOCHS = 8
PATIENCE = 2
LR = 1e-4
SEED = 42              # used for the data split, so it matches V2
VAL_FRACTION = 0.1
TIME_BUDGET_PER_MODEL_H = 1.3
RUNS = [("densenet121", 1), ("densenet121", 2), ("efficientnet_b0", 42)]
LABELS = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass", "Nodule", "Pneumonia",
    "Pneumothorax", "Consolidation", "Edema", "Emphysema", "Fibrosis", "Pleural_Thickening", "Hernia",
]
OUT = Path(os.environ.get("OUT_DIR", "/kaggle/working/results_v21"))
CACHE = Path(os.environ.get("CACHE_DIR", "/tmp/cxr_cache"))
INPUT_ROOT = os.environ.get("INPUT_ROOT", "/kaggle/input")
OUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

## 1. Find the data and build the patient-wise split

In [2]:
def find_one(name: str) -> Path:
    hits = glob.glob(os.path.join(INPUT_ROOT, "**", name), recursive=True)
    if not hits:
        raise FileNotFoundError(name + " not found under " + INPUT_ROOT + ". Add the nih-chest-xrays/data dataset.")
    return Path(hits[0])


def index_images(root: str) -> dict:
    paths = {}
    for dirpath, _, files in os.walk(root):
        for f in files:
            if f.endswith(".png"):
                paths[f] = os.path.join(dirpath, f)
    return paths


def multi_hot(finding_labels: pd.Series) -> np.ndarray:
    y = np.zeros((len(finding_labels), len(LABELS)), dtype=np.float32)
    pos = {name: i for i, name in enumerate(LABELS)}
    for row, text in enumerate(finding_labels):
        for name in str(text).split("|"):
            if name in pos:
                y[row, pos[name]] = 1.0
    return y


def patient_split(df: pd.DataFrame, fraction: float, seed: int) -> tuple:
    patients = np.array(sorted(df["Patient ID"].unique()))
    rng = np.random.default_rng(seed)
    rng.shuffle(patients)
    n_val = max(1, int(round(len(patients) * fraction)))
    val_patients = set(patients[:n_val].tolist())
    is_val = df["Patient ID"].isin(val_patients).to_numpy()
    return df[~is_val].reset_index(drop=True), df[is_val].reset_index(drop=True)


def load_splits() -> tuple:
    entry = find_one("Data_Entry_2017.csv")
    root = entry.parent
    df = pd.read_csv(entry)
    images = index_images(str(root))
    df = df[df["Image Index"].isin(images)].copy()
    df["path"] = df["Image Index"].map(images)
    train_val_names = set(find_one("train_val_list.txt").read_text().split())
    test_names = set(find_one("test_list.txt").read_text().split())
    train_val = df[df["Image Index"].isin(train_val_names)].reset_index(drop=True)
    test = df[df["Image Index"].isin(test_names)].reset_index(drop=True)
    train, val = patient_split(train_val, VAL_FRACTION, SEED)
    if SMOKE:
        train, val, test = train.head(2000), val.head(500), test.head(500)
    shared = set(train["Patient ID"]) & (set(val["Patient ID"]) | set(test["Patient ID"]))
    assert not (set(train["Patient ID"]) & set(val["Patient ID"])), "patient leakage train/val"
    assert not (set(train_val["Patient ID"]) & set(test["Patient ID"])), "patient leakage train_val/test"
    print("images found:", len(df), "| train", len(train), "val", len(val), "test", len(test),
          "| patients shared train vs val/test:", len(shared))
    return train, val, test

## 2. Resize every image once to 224x224 (grayscale) and cache it
Decoding 1024x1024 PNGs every epoch is the slowest part, so we do it once.

In [3]:
def _load_small(path: str) -> np.ndarray:
    from PIL import Image
    with Image.open(path) as im:
        return np.asarray(im.convert("L").resize((IMG, IMG), Image.BILINEAR), dtype=np.uint8)


def build_cache(df: pd.DataFrame, name: str) -> Path:
    file = CACHE / (name + "_" + str(IMG) + "_" + str(len(df)) + ".npy")
    if file.exists():
        return file
    arr = np.lib.format.open_memmap(str(file), mode="w+", dtype=np.uint8, shape=(len(df), IMG, IMG))
    t0 = time.time()
    with Pool(os.cpu_count() or 2) as pool:
        for i, img in enumerate(pool.imap(_load_small, df["path"].tolist(), chunksize=64)):
            arr[i] = img
            if i % 10000 == 0:
                print(name, i, "/", len(df), round(time.time() - t0), "s")
    arr.flush()
    del arr
    return file

## 3. Metrics used for every model

In [4]:
def per_class_auroc(y: np.ndarray, p: np.ndarray) -> list:
    from sklearn.metrics import roc_auc_score
    return [float(roc_auc_score(y[:, k], p[:, k])) if 0 < y[:, k].sum() < len(y) else float("nan")
            for k in range(y.shape[1])]


def ece(y: np.ndarray, p: np.ndarray, bins: int = 15) -> float:
    edges = np.linspace(0, 1, bins + 1)
    which = np.clip(np.digitize(p, edges[1:-1]), 0, bins - 1)
    return float(sum((which == b).mean() * abs(y[which == b].mean() - p[which == b].mean())
                     for b in range(bins) if (which == b).any()))


def platt_fit_apply(y_fit: np.ndarray, p_fit: np.ndarray, p_apply: np.ndarray) -> np.ndarray:
    from sklearn.linear_model import LogisticRegression
    def logit(p):
        p = np.clip(p, 1e-6, 1 - 1e-6)
        return np.log(p / (1 - p))
    out = np.zeros_like(p_apply)
    for k in range(y_fit.shape[1]):
        lr = LogisticRegression(C=1e6, max_iter=1000)
        lr.fit(logit(p_fit[:, k:k + 1]), y_fit[:, k])
        out[:, k] = lr.predict_proba(logit(p_apply[:, k:k + 1]))[:, 1]
    return out


def patient_bootstrap(y: np.ndarray, p: np.ndarray, patients: np.ndarray, n: int = 200) -> list:
    rng = np.random.default_rng(SEED)
    groups = pd.Series(np.arange(len(y))).groupby(patients).apply(lambda s: s.to_numpy()).tolist()
    means = []
    for _ in range(n):
        pick = rng.integers(0, len(groups), len(groups))
        idx = np.concatenate([groups[i] for i in pick])
        means.append(np.nanmean(per_class_auroc(y[idx], p[idx])))
    return [float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))]


def subgroups(df: pd.DataFrame, y: np.ndarray, p: np.ndarray) -> list:
    age = pd.to_numeric(df["Patient Age"], errors="coerce").to_numpy()
    sex = df["Patient Gender"].astype(str).to_numpy()
    masks = {"sex=M": sex == "M", "sex=F": sex == "F",
             "age<40": age < 40, "age 40-59": (age >= 40) & (age < 60), "age 60-100": (age >= 60) & (age <= 100)}
    rows = []
    for name, m in masks.items():
        rows.append({"group": name, "n_images": int(m.sum()), "mean_auroc": float(np.nanmean(per_class_auroc(y[m], p[m])))})
    return rows

## 4. Train each run, save validation and test predictions

In [5]:
def train_and_predict(train_df, val_df, test_df) -> dict:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, Dataset
    from torchvision import models

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")
    caches = {n: build_cache(d, n) for n, d in [("train", train_df), ("val", val_df), ("test", test_df)]}
    targets = {"train": multi_hot(train_df["Finding Labels"]), "val": multi_hot(val_df["Finding Labels"]),
               "test": multi_hot(test_df["Finding Labels"])}

    class CachedXray(Dataset):
        def __init__(self, file, y):
            self.file, self.y, self.arr = str(file), y, None

        def __len__(self):
            return len(self.y)

        def __getitem__(self, i):
            if self.arr is None:
                self.arr = np.load(self.file, mmap_mode="r")
            return torch.from_numpy(np.array(self.arr[i])), torch.from_numpy(self.y[i])

    workers = min(4, os.cpu_count() or 2)
    mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

    def prepare(x, train):
        x = x.to(device, non_blocking=True).float().div_(255.0).unsqueeze(1).repeat(1, 3, 1, 1)
        if train:
            flip = torch.rand(x.shape[0], device=device) < 0.5
            x[flip] = x[flip].flip(-1)
        return (x - mean) / std

    def build(name):
        if name == "densenet121":
            m = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
            m.classifier = nn.Linear(m.classifier.in_features, len(LABELS))
        elif name == "efficientnet_b0":
            m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
            m.classifier[1] = nn.Linear(m.classifier[1].in_features, len(LABELS))
        else:
            raise ValueError(name)
        return m.to(device)

    @torch.no_grad()
    def predict(model, loader):
        model.eval()
        out = []
        for x, _ in loader:
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                out.append(torch.sigmoid(model(prepare(x, False))).float().cpu().numpy())
        return np.concatenate(out)

    pos = targets["train"].sum(0)
    pos_weight = torch.tensor(np.clip((len(targets["train"]) - pos) / np.maximum(pos, 1), 1, 50),
                              dtype=torch.float32, device=device)
    eval_loaders = {s: DataLoader(CachedXray(caches[s], targets[s]), batch_size=BATCH, shuffle=False,
                                  num_workers=workers, pin_memory=True) for s in ["val", "test"]}
    done = {}
    for name, seed in RUNS:
        tag = name + "_seed" + str(seed)
        try:
            torch.manual_seed(seed)
            np.random.seed(seed)
            gen = torch.Generator()
            gen.manual_seed(seed)
            train_loader = DataLoader(CachedXray(caches["train"], targets["train"]), batch_size=BATCH, shuffle=True,
                                      num_workers=workers, pin_memory=True, generator=gen)
            model = build(name)
            opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
            sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=0)
            scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
            loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
            best, bad, history, t0 = -1.0, 0, [], time.time()
            weights_file = OUT / (tag + "_best.pt")
            for epoch in range(1 if SMOKE else MAX_EPOCHS):
                model.train()
                total, seen = 0.0, 0
                for x, y in train_loader:
                    y = y.to(device, non_blocking=True)
                    opt.zero_grad(set_to_none=True)
                    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                        loss = loss_fn(model(prepare(x, True)), y)
                    scaler.scale(loss).backward()
                    scaler.step(opt)
                    scaler.update()
                    total += loss.item() * len(y)
                    seen += len(y)
                val_auc = float(np.nanmean(per_class_auroc(targets["val"], predict(model, eval_loaders["val"]))))
                sched.step(val_auc)
                hours = (time.time() - t0) / 3600
                history.append({"run": tag, "epoch": epoch + 1, "train_loss": total / seen,
                                "val_mean_auroc": val_auc, "hours": round(hours, 3)})
                print(history[-1], flush=True)
                if val_auc > best:
                    best, bad = val_auc, 0
                    torch.save(model.state_dict(), weights_file)
                else:
                    bad += 1
                if bad >= PATIENCE or hours > TIME_BUDGET_PER_MODEL_H:
                    break
            model.load_state_dict(torch.load(weights_file, map_location=device))
            val_p, test_p = predict(model, eval_loaders["val"]), predict(model, eval_loaders["test"])
            np.save(OUT / (tag + "_val_probs.npy"), val_p.astype(np.float16))
            np.save(OUT / (tag + "_test_probs.npy"), test_p.astype(np.float16))
            pd.DataFrame(history).to_csv(OUT / (tag + "_history.csv"), index=False)
            done[tag] = {"val": val_p, "test": test_p, "history": history, "model": model}
            if tag != "densenet121_seed1":
                del done[tag]["model"]
            if device.type == "cuda":
                torch.cuda.empty_cache()
        except Exception:
            print("RUN FAILED:", tag)
            traceback.print_exc()
    np.save(OUT / "val_targets.npy", targets["val"].astype(np.uint8))
    np.save(OUT / "test_targets.npy", targets["test"].astype(np.uint8))
    return {"runs": done, "targets": targets, "caches": caches, "prepare": prepare}

## 5. Run training (about 2.5 to 3.5 hours on a T4)

In [6]:
train_df, val_df, test_df = load_splits()
trained = train_and_predict(train_df, val_df, test_df)

images found: 112120 | train 77719 val 8805 test 25596 | patients shared train vs val/test: 0
device: cuda Tesla T4
train 0 / 77719 24 s
train 10000 / 77719 694 s
train 20000 / 77719 1348 s
train 30000 / 77719 2009 s
train 40000 / 77719 2680 s
train 50000 / 77719 3356 s
train 60000 / 77719 4030 s
train 70000 / 77719 4721 s
val 0 / 8805 18 s
test 0 / 25596 17 s
test 10000 / 25596 703 s
test 20000 / 25596 1381 s
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 184MB/s]


{'run': 'densenet121_seed1', 'epoch': 1, 'train_loss': 0.9582708807406576, 'val_mean_auroc': 0.8124087944941742, 'hours': 0.101}
{'run': 'densenet121_seed1', 'epoch': 2, 'train_loss': 0.8462266697571805, 'val_mean_auroc': 0.827572923323243, 'hours': 0.201}
{'run': 'densenet121_seed1', 'epoch': 3, 'train_loss': 0.7875052696441981, 'val_mean_auroc': 0.8277619089094778, 'hours': 0.302}
{'run': 'densenet121_seed1', 'epoch': 4, 'train_loss': 0.7366848129215267, 'val_mean_auroc': 0.829073536885993, 'hours': 0.403}
{'run': 'densenet121_seed1', 'epoch': 5, 'train_loss': 0.6782170909589719, 'val_mean_auroc': 0.8295387999562023, 'hours': 0.503}
{'run': 'densenet121_seed1', 'epoch': 6, 'train_loss': 0.6229876914070246, 'val_mean_auroc': 0.8263923764940229, 'hours': 0.604}
{'run': 'densenet121_seed1', 'epoch': 7, 'train_loss': 0.5118815925691623, 'val_mean_auroc': 0.8228435515191641, 'hours': 0.704}
{'run': 'densenet121_seed2', 'epoch': 1, 'train_loss': 0.9617156167449015, 'val_mean_auroc': 0.8128

100%|██████████| 20.5M/20.5M [00:00<00:00, 170MB/s]


{'run': 'efficientnet_b0_seed42', 'epoch': 1, 'train_loss': 1.0058749384661279, 'val_mean_auroc': 0.7966479940369277, 'hours': 0.061}
{'run': 'efficientnet_b0_seed42', 'epoch': 2, 'train_loss': 0.8827733944631135, 'val_mean_auroc': 0.817791144177202, 'hours': 0.116}
{'run': 'efficientnet_b0_seed42', 'epoch': 3, 'train_loss': 0.8233446959521608, 'val_mean_auroc': 0.8227801327895717, 'hours': 0.171}
{'run': 'efficientnet_b0_seed42', 'epoch': 4, 'train_loss': 0.7715377385470985, 'val_mean_auroc': 0.8198219471781013, 'hours': 0.226}
{'run': 'efficientnet_b0_seed42', 'epoch': 5, 'train_loss': 0.7105160893864927, 'val_mean_auroc': 0.8238380227519768, 'hours': 0.28}
{'run': 'efficientnet_b0_seed42', 'epoch': 6, 'train_loss': 0.6725291559089529, 'val_mean_auroc': 0.8230906703062735, 'hours': 0.335}
{'run': 'efficientnet_b0_seed42', 'epoch': 7, 'train_loss': 0.6343055926665488, 'val_mean_auroc': 0.8224533322693456, 'hours': 0.39}


## 6. Evaluate: patient-level intervals, validation-fitted calibration, subgroups

**Note on the referral numbers below.** In the run whose output is saved here, the line that orders images by confidence ended with `[::-1]`, so the code kept the *least* confident images instead of the most confident. That reversed the referral result (`selective_mean_auroc_calibrated` and `selective_calibrated_v21.csv`). The line is fixed in the code below; the corrected numbers come from re-reading the saved predictions with `analyze_v21.py` and are in `analysis/referral.csv`. Everything else in this run is unaffected.

In [7]:
def evaluate(trained) -> None:
    y_val, y_test = trained["targets"]["val"], trained["targets"]["test"]
    patients = test_df["Patient ID"].to_numpy()
    summary, per_class_rows, sub_rows, sel_rows = {}, [], [], []
    for tag, r in trained["runs"].items():
        try:
            raw = r["test"].astype(np.float64)
            cal = platt_fit_apply(y_val, r["val"].astype(np.float64), raw)
            q = np.clip(cal, 1e-6, 1 - 1e-6)
            order = np.argsort(-(q * np.log(q) + (1 - q) * np.log(1 - q)).mean(1))  # most confident first
            sel = {}
            for c in (1.0, 0.9, 0.8, 0.7, 0.5):
                keep = order[: int(round(c * len(raw)))]
                sel[str(c)] = float(np.nanmean(per_class_auroc(y_test[keep], cal[keep])))
                sel_rows.append({"run": tag, "coverage": c, "mean_auroc_kept": sel[str(c)]})
            scores = per_class_auroc(y_test, raw)
            summary[tag] = {
                "test_mean_auroc": float(np.nanmean(scores)),
                "patient_bootstrap_ci95": patient_bootstrap(y_test, raw, patients),
                "best_val_mean_auroc": max(h["val_mean_auroc"] for h in r["history"]),
                "epochs_run": len(r["history"]),
                "train_hours": r["history"][-1]["hours"],
                "mean_ece_raw": float(np.mean([ece(y_test[:, k], raw[:, k]) for k in range(14)])),
                "mean_ece_val_calibrated": float(np.mean([ece(y_test[:, k], cal[:, k]) for k in range(14)])),
                "mean_brier_raw": float(np.mean((raw - y_test) ** 2)),
                "mean_brier_val_calibrated": float(np.mean((cal - y_test) ** 2)),
                "selective_mean_auroc_calibrated": sel,
            }
            per_class_rows += [{"run": tag, "label": l, "test_auroc": s} for l, s in zip(LABELS, scores)]
            sub_rows += [dict(run=tag, **row) for row in subgroups(test_df, y_test, raw)]
            print(tag, json.dumps(summary[tag]), flush=True)
        except Exception:
            print("EVAL FAILED:", tag)
            traceback.print_exc()
    seeds = [v["test_mean_auroc"] for k, v in summary.items() if k.startswith("densenet121")]
    run_info = {"smoke": SMOKE, "img": IMG, "batch": BATCH, "lr": LR, "split_seed": SEED, "runs": [list(r) for r in RUNS],
                "n_train": len(train_df), "n_val": len(val_df), "n_test": len(test_df),
                "densenet121_new_seed_test_aurocs": seeds}
    (OUT / "summary_v21.json").write_text(json.dumps({"run": run_info, "models": summary}, indent=2))
    pd.DataFrame(per_class_rows).to_csv(OUT / "per_class_auroc_v21.csv", index=False)
    pd.DataFrame(sub_rows).to_csv(OUT / "subgroups_v21.csv", index=False)
    pd.DataFrame(sel_rows).to_csv(OUT / "selective_calibrated_v21.csv", index=False)
    print(pd.DataFrame(sub_rows).pivot(index="group", columns="run", values="mean_auroc").round(3))


evaluate(trained)

densenet121_seed1 {"test_mean_auroc": 0.8007678975400203, "patient_bootstrap_ci95": [0.795068402130931, 0.8063140364889245], "best_val_mean_auroc": 0.8295387999562023, "epochs_run": 7, "train_hours": 0.704, "mean_ece_raw": 0.2761758570510677, "mean_ece_val_calibrated": 0.010636337204563412, "mean_brier_raw": 0.18511201408150502, "mean_brier_val_calibrated": 0.05821415326417779, "selective_mean_auroc_calibrated": {"1.0": 0.8007678975400203, "0.9": 0.7915545226397073, "0.8": 0.7858159891140464, "0.7": 0.7813412675384279, "0.5": 0.7705453484353121}}
densenet121_seed2 {"test_mean_auroc": 0.8027993653059894, "patient_bootstrap_ci95": [0.7966543315756643, 0.8082859072755005], "best_val_mean_auroc": 0.8309079385994028, "epochs_run": 7, "train_hours": 0.704, "mean_ece_raw": 0.26138282193750595, "mean_ece_val_calibrated": 0.011479295798015838, "mean_brier_raw": 0.1744210629339195, "mean_brier_val_calibrated": 0.05813505795987376, "selective_mean_auroc_calibrated": {"1.0": 0.8027993653059894, "0

## 7. Grad-CAM for a few test images (DenseNet121, seed 1)
Shows where the model looked for its highest-scoring correct finding. A heatmap is a hint, not an explanation.

In [8]:
def gradcam_grid(trained, n: int = 6) -> None:
    import torch
    import torch.nn.functional as F
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    run = trained["runs"].get("densenet121_seed1")
    if run is None or "model" not in run:
        print("no DenseNet121 seed 1 model, skipping Grad-CAM")
        return
    model = run["model"].eval()
    y, p = trained["targets"]["test"], run["test"]
    arr = np.load(trained["caches"]["test"], mmap_mode="r")
    picks = []
    for k in [LABELS.index(l) for l in ["Cardiomegaly", "Effusion", "Pneumothorax", "Mass", "Edema", "Emphysema"]][:n]:
        cand = np.where(y[:, k] == 1)[0]
        picks.append((int(cand[np.argmax(p[cand, k])]), k))
    fig, axes = plt.subplots(2, (len(picks) + 1) // 2, figsize=(4 * ((len(picks) + 1) // 2), 8))
    for ax, (i, k) in zip(np.ravel(axes), picks):
        x = trained["prepare"](torch.from_numpy(np.array(arr[i]))[None], False).requires_grad_(False)
        feats = F.relu(model.features(x))
        feats.retain_grad()
        logits = model.classifier(F.adaptive_avg_pool2d(feats, 1).flatten(1))
        model.zero_grad()
        logits[0, k].backward()
        w = feats.grad.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * feats).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=(IMG, IMG), mode="bilinear", align_corners=False)[0, 0].detach().cpu().numpy()
        cam = cam / (cam.max() + 1e-8)
        ax.imshow(np.array(arr[i]), cmap="gray")
        ax.imshow(cam, cmap="jet", alpha=0.35)
        ax.set_title(LABELS[k] + " (p=" + str(round(float(p[i, k]), 2)) + ")")
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT / "gradcam_examples.png", dpi=130)
    plt.close()
    print("saved Grad-CAM grid")


try:
    gradcam_grid(trained)
except Exception:
    print("GRAD-CAM FAILED")
    traceback.print_exc()

saved Grad-CAM grid
